# Oxygen on Cu(111): surface molecular dynamics
Continue the oxygen-adsorption exercise using **MACE-MP medium**, the same materials
model and 2 x 2 x 3 Cu slab. Relax one adsorption site and run short ASE Langevin MD.
Oxygen and the upper Cu layers move; the bottom layer remains fixed.

Use the MACE kernel. This small periodic cell represents one O per four surface Cu
atoms, not an isolated dilute adsorbate. It is a workshop example, not a converged
prediction of diffusion or desorption rates.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from ase import units
from ase.build import fcc111, add_adsorbate
from ase.constraints import FixAtoms
from ase.optimize import BFGS
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import MACECalculator
sys.path.insert(0, str(Path.cwd().parent.parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.view_mols import view
_, OUTPUT = start_exercise()
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '4')))
calc = MACECalculator(model_paths='../../models/2023-12-03-mace-128-L1_epoch-199.model',
                      device='cpu', default_dtype='float64')

## 1. Prepare and relax
Choose `SITE` from fcc, hcp, bridge, or ontop. Relaxation may move oxygen away from
its starting site. Periodicity is in the surface plane; the normal direction has
10 angstrom of vacuum on each side of the original slab.

In [ ]:
SITE = 'fcc'
atoms = fcc111('Cu', size=(2, 2, 3), vacuum=10.0)
bottom = np.flatnonzero(atoms.get_tags() == atoms.get_tags().max())
top = np.flatnonzero(atoms.get_tags() == 1)
cu = np.arange(len(atoms))
add_adsorbate(atoms, 'O', height=1.5, position=SITE)
oxygen = len(atoms) - 1
atoms.set_constraint(FixAtoms(indices=bottom))
atoms.calc = calc
assert BFGS(atoms, logfile=str(OUTPUT/'relax.log')).run(fmax=0.03, steps=300)
reference = atoms.copy()
write(OUTPUT/'relaxed.extxyz', atoms)
view(atoms)

## 2. Run a short trajectory
Use the temperature, timestep, and step count set below.
The simulated duration is `STEPS * DT_FS / 1000` ps. The thermostat acts on the mobile
atoms; fixed atoms retain zero velocity. The fixed substrate anchors the system,
so do not separately remove center-of-mass motion. Total molecular energy need
not be conserved under Langevin dynamics.

To change temperature or site for an independent run, rerun the preparation cell
first. Save every 5 steps. No wrapping is applied during MD, so coordinates retain
continuous lateral displacements across periodic cell boundaries.

In [ ]:
TEMPERATURE_K = 900
DT_FS = 0.5
STEPS = 1000
SAVE_EVERY = 5
rng = np.random.default_rng(42)
MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE_K, rng=rng)
dyn = Langevin(atoms, DT_FS*units.fs, temperature_K=TEMPERATURE_K,
               friction=0.01/units.fs, fixcm=False, rng=rng,
               trajectory=str(OUTPUT/'oxygen_copper.traj'),
               logfile=str(OUTPUT/'md.log'), loginterval=SAVE_EVERY)
start = time.perf_counter()
dyn.run(STEPS)
print(f'Elapsed: {(time.perf_counter()-start)/60:.1f} min')

## 3. Follow the adsorbate and surface
Height is measured relative to the mean z coordinate of the initially top-layer Cu
atoms. The nearest O-Cu distance uses periodic minimum images. Lateral displacement
is relative to the relaxed starting position; top-layer RMS displacement measures
surface motion. Neither an excursion nor a short animation establishes a diffusion
coefficient. Bond sticks in the animation use the starting geometry and are visual guides,
not a time-dependent chemical bond assignment.

In [ ]:
frames = read(OUTPUT/'oxygen_copper.traj', index=':')
t = np.arange(len(frames))*SAVE_EVERY*DT_FS/1000
xyz = np.array([frame.positions for frame in frames])
height = xyz[:, oxygen, 2] - xyz[:, top, 2].mean(axis=1)
lateral = np.linalg.norm(xyz[:, oxygen, :2]-reference.positions[oxygen, :2], axis=1)
nearest = np.array([frame.get_distances(oxygen, cu, mic=True).min() for frame in frames])
surface_rms = np.sqrt(np.mean(np.sum((xyz[:, top]-reference.positions[top])**2, axis=2), axis=1))
fig, ax = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
ax[0].plot(t, height, label='Height above top layer')
ax[0].plot(t, nearest, label='Nearest O-Cu'); ax[0].set_ylabel('Distance (angstrom)'); ax[0].legend()
ax[1].plot(t, lateral, label='O lateral displacement')
ax[1].plot(t, surface_rms, label='Top Cu layer RMS displacement')
ax[1].set_ylabel('Displacement (angstrom)'); ax[1].legend()
ax[2].plot(t, [frame.get_temperature() for frame in frames])
ax[2].axhline(TEMPERATURE_K, color='gray', linestyle='--')
ax[2].set(xlabel='Time (ps)', ylabel='Instantaneous T (K)')
plt.tight_layout(); plt.show()
np.savetxt(OUTPUT/'surface_motion.csv', np.column_stack((t, height, nearest, lateral, surface_rms)),
           delimiter=',', header='time_ps,height_A,nearest_OCu_A,lateral_A,top_Cu_rms_A', comments='')
view(frames[::max(1, len(frames)//100)], animate=True)

In [ ]:
from ase.visualize import view as ase_view
ase_view(frames[-1])

## Try and explain
- Does oxygen vibrate near one site, or move between sites? What evidence distinguishes them?
- Repeat at another temperature or starting site. Is the sampling long enough to compare populations?
- How do the fixed layer, small periodic cell, and thermostat affect the interpretation?